# Skripsi.ipynb — Training & Evaluasi YOLO (deteksi plat)
Environment: `.venv` (Python 3.14, torch cu126). Lihat `../INSTALL.md`.
Alur: cek GPU → download dataset → training baseline → evaluasi (PT vs ONNX) → dataset pembanding → dataset gabungan → training ulang → evaluasi akhir.
Semua path relatif ke folder `Code/` (`../runs`, `../dataset_roboflow`, dst). OCR ada di `Pipeline.ipynb`.

## Peta isi notebook (Skripsi.ipynb, `.venv` Python 3.14)

| Bagian | Judul di bawah |
|---|---|
| Cek GPU | 1. Cek GPU |
| Download dataset | 2-3. API key Roboflow, dataset utama (kamera jalan) |
| Data & konfigurasi | 4. Cek jumlah data, `data_fixed.yaml` |
| Training tahap 1 | 5. Training baseline YOLOv8n |
| Evaluasi PT vs ONNX | 6-7. Evaluasi baseline, export ONNX, PT vs ONNX |
| Dataset pembanding | 8-10. Denny (usang), Aldian, cek label |
| Dataset gabungan | 11. Penggabungan dataset |
| Evaluasi model lama | 12. Baseline pada dua test set |
| Training tahap 2 | 13. Training ulang pada dataset gabungan |
| Evaluasi tahap 2 & uji foto | 14. Evaluasi model baru dan foto asli |

Output sel evaluasi adalah bukti hasil untuk Bab 4, jangan dihapus.

## 1. Cek GPU
Memastikan PyTorch memakai CUDA (RTX 4050).

In [1]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

2.14.1+cu126
True
NVIDIA GeForce RTX 4050 Laptop GPU


## 2. API key Roboflow
Membaca `ROBOFLOW_API_KEY` dari `../.env`.

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()
key = os.getenv("ROBOFLOW_API_KEY")
print("Key kebaca:", key is not None)

Key kebaca: True


## 3. Download dataset utama (kamera jalan)
Dataset plat Indonesia dari workspace sendiri → `../dataset_roboflow`.

In [3]:
from roboflow import Roboflow

rf = Roboflow(api_key=os.getenv("ROBOFLOW_API_KEY"))
project = rf.workspace("muhammad-rizky-ramadhan-e5hve").project("license-plate-indonesia-8coz5")
dataset = project.version(1).download("yolov8", location="../dataset_roboflow")
print(dataset.location)

loading Roboflow workspace...
loading Roboflow project...
Exporting format yolov8 in progress : 95.0%
Version export complete for yolov8 format



Extracting Dataset Version Zip to ../dataset_roboflow in yolov8:: 100%|██████████| 2642/2642 [00:01<00:00, 1534.12it/s]


Creating new Ultralytics Settings v0.0.8 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\riizk\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
c:\Users\riizk\Documents\3.Collage\SKRIPSI\dataset_roboflow


## 4. Cek jumlah data & buat `data_fixed.yaml`
Hitung gambar/label per split, lalu buat yaml dengan path absolut untuk training.

In [4]:
import glob

for split in ["train", "valid", "test"]:
    n_img = len(glob.glob(f"{dataset.location}/{split}/images/*"))
    n_lbl = len(glob.glob(f"{dataset.location}/{split}/labels/*.txt"))
    print(split, "-> gambar:", n_img, "| label:", n_lbl)

print()
print(open(f"{dataset.location}/data.yaml").read())

train -> gambar: 1098 | label: 1098
valid -> gambar: 167 | label: 167
test -> gambar: 54 | label: 54

names:
- plate
nc: 1
roboflow:
  license: CC BY 4.0
  project: license-plate-indonesia-8coz5
  url: https://app.roboflow.com/muhammad-rizky-ramadhan-e5hve/license-plate-indonesia-8coz5/1
  version: 1
  workspace: muhammad-rizky-ramadhan-e5hve
test: ../test/images
train: ../train/images
val: ../valid/images



In [5]:
import yaml
from pathlib import Path

root = Path(dataset.location).resolve()
cfg = {
    "path": str(root),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": 1,
    "names": ["plate"],
}
data_yaml = root / "data_fixed.yaml"
with open(data_yaml, "w") as f:
    yaml.dump(cfg, f)
print(data_yaml)

C:\Users\riizk\Documents\3.Collage\SKRIPSI\dataset_roboflow\data_fixed.yaml


## 5. Training baseline YOLOv8n
100 epoch (patience 20, `fliplr=0` karena teks plat tidak boleh dicerminkan) → `../runs/slpr_v8n_run1`.

In [7]:
model = YOLO("yolov8n.pt")
model.train(
    data=str(data_yaml),
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    fliplr=0.0,
    device=0,
    workers=2,
    project=str(root.parent / "runs"),
    name="slpr_v8n_run1",
)

Ultralytics 8.4.173  Python-3.14.5 torch-2.14.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\riizk\Documents\3.Collage\SKRIPSI\dataset_roboflow\data_fixed.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x00000255CE1346E0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0480

## 6. Evaluasi baseline pada test set
Precision, recall, mAP50, mAP50-95 → `../runs/eval_test`.

In [8]:
from ultralytics import YOLO

best = root.parent / "runs" / "slpr_v8n_run1" / "weights" / "best.pt"
print("best.pt ada:", best.exists())

model = YOLO(str(best))
metrics = model.val(
    data=str(data_yaml),
    split="test",
    imgsz=640,
    device=0,
    project=str(root.parent / "runs"),
    name="eval_test",
)

print("precision :", metrics.box.mp)
print("recall    :", metrics.box.mr)
print("mAP50     :", metrics.box.map50)
print("mAP50-95  :", metrics.box.map)

best.pt ada: True
Ultralytics 8.4.173  Python-3.14.5 torch-2.14.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
WARNING val: Slow image access detected (ping: 0.10.0 ms, read: 5.81.5 MB/s, size: 33.0 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning C:\Users\riizk\Documents\3.Collage\SKRIPSI\dataset_roboflow\test\labels... 54 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 54/54 844.0it/s 0.1s
val: New cache created: C:\Users\riizk\Documents\3.Collage\SKRIPSI\dataset_roboflow\test\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 1.3it/s 3.1s0.3ss
                   all         54         54      0.998      0.981      0.995      0.688
Speed: 1.6ms preprocess, 7.2ms inference, 0.0ms loss, 1.8ms postprocess per imag

## 7. Export ONNX & perbandingan PT vs ONNX
Export `best.pt` ke ONNX, lalu evaluasi ONNX (CPU) dan PT (CPU) → `eval_test_onnx`, `eval_test_pt_cpu`.

In [11]:
onnx_path = model.export(format="onnx", imgsz=640, simplify=True)
print(onnx_path)

Ultralytics 8.4.173  Python-3.14.5 torch-2.14.1+cu126 CPU (AMD Ryzen 7 8845HS w/ Radeon 780M Graphics)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from 'C:\Users\riizk\Documents\3.Collage\SKRIPSI\runs\slpr_v8n_run1\weights\best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 5, 8400) (6.0 MB)

ONNX: starting export with onnx 1.23.1 opset 18...
ONNX: slimming with onnxslim 0.1.97...
ONNX: export success  0.6s, saved as 'C:\Users\riizk\Documents\3.Collage\SKRIPSI\runs\slpr_v8n_run1\weights\best.onnx' (11.7 MB)

Export complete (0.7s)
Results saved to C:\Users\riizk\Documents\3.Collage\SKRIPSI\runs\slpr_v8n_run1\weights\best.onnx
Predict:         yolo predict task=detect model=C:\Users\riizk\Documents\3.Collage\SKRIPSI\runs\slpr_v8n_run1\weights\best.onnx imgsz=640 
Validate:        yolo val task=detect model=C:\Users\riizk\Documents\3.Collage\SKRIPSI\runs\slpr_v8n_run1\weights\best.onnx imgsz=640 data=C:\Users\riiz

In [12]:
from pathlib import Path
from ultralytics import YOLO

root = Path("../dataset_roboflow").resolve()
data_yaml = root / "data_fixed.yaml"
onnx_path = root.parent / "runs" / "slpr_v8n_run1" / "weights" / "best.onnx"
print("onnx ada:", onnx_path.exists())

model_onnx = YOLO(str(onnx_path), task="detect")
m = model_onnx.val(
    data=str(data_yaml),
    split="test",
    imgsz=640,
    device="cpu",
    project=str(root.parent / "runs"),
    name="eval_test_onnx",
)
print("precision:", m.box.mp, "| recall:", m.box.mr)
print("mAP50:", m.box.map50, "| mAP50-95:", m.box.map)

onnx ada: True
Ultralytics 8.4.173  Python-3.14.5 torch-2.14.1+cu126 CPU (AMD Ryzen 7 8845HS w/ Radeon 780M Graphics)
Loading C:\Users\riizk\Documents\3.Collage\SKRIPSI\runs\slpr_v8n_run1\weights\best.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.30.0 with CPUExecutionProvider
Setting batch=1 input of shape (1, 3, 640, 640)
val: Fast image access  (ping: 0.10.0 ms, read: 380.3139.4 MB/s, size: 40.2 KB)
val: Scanning C:\Users\riizk\Documents\3.Collage\SKRIPSI\dataset_roboflow\test\labels.cache... 54 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 54/54 25.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 54/54 26.5it/s 2.0s0.1s
                   all         54         54      0.964      0.999      0.976       0.66
Speed: 0.9ms preprocess, 29.6ms inference, 0.0ms loss, 1.1ms postprocess per image
Results saved to C:\Users\riizk\Documents\3.Collage\SKRIPSI\runs\eval_test_onnx
precision: 0.9642667579671631

In [13]:
from pathlib import Path
from ultralytics import YOLO

root = Path("../dataset_roboflow").resolve()
data_yaml = root / "data_fixed.yaml"
best_pt = root.parent / "runs" / "slpr_v8n_run1" / "weights" / "best.pt"

pt = YOLO(str(best_pt))
m_pt = pt.val(
    data=str(data_yaml),
    split="test",
    imgsz=640,
    device="cpu",
    batch=1,
    project=str(root.parent / "runs"),
    name="eval_test_pt_cpu",
)
print("PT CPU -> P:", m_pt.box.mp, "| R:", m_pt.box.mr)
print("mAP50:", m_pt.box.map50, "| mAP50-95:", m_pt.box.map)

Ultralytics 8.4.173  Python-3.14.5 torch-2.14.1+cu126 CPU (AMD Ryzen 7 8845HS w/ Radeon 780M Graphics)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 285.144.8 MB/s, size: 34.4 KB)
val: Scanning C:\Users\riizk\Documents\3.Collage\SKRIPSI\dataset_roboflow\test\labels.cache... 54 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 54/54 16.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 54/54 25.5it/s 2.1s0.0s
                   all         54         54      0.998      0.981      0.995      0.688
Speed: 0.5ms preprocess, 32.9ms inference, 0.0ms loss, 1.1ms postprocess per image
Results saved to C:\Users\riizk\Documents\3.Collage\SKRIPSI\runs\eval_test_pt_cpu
PT CPU -> P: 0.9977242490720883 | R: 0.9814814814814815
mAP50: 0.994642857142857 | mAP50-95: 0.6883453517251973


## 8. Dataset pembanding 1 (Denny)
Kode download sebagai dokumentasi sumber data. Folder `dataset_denny_v2` sudah dihapus (tidak dipakai di training); jalankan sel ini lagi kalau mau mengunduh ulang.

> Sel di bagian ini memakai dataset_denny_v2 yang sudah tidak ada. **Hanya untuk dilihat, jangan dijalankan ulang.** Dataset Denny hanya dipakai untuk pengujian awal, bukan training.

In [ ]:
import os
from dotenv import load_dotenv
from roboflow import Roboflow

load_dotenv()
rf = Roboflow(api_key=os.getenv("ROBOFLOW_API_KEY"))
proj = rf.workspace("denny-pmzkg").project("plat-nomor-kendaraan-indonesia")

for v in (2, 1):
    try:
        ds = proj.version(v).download("yolov8", location=f"../dataset_denny_v{v}")
        print("OK versi", v, ds.location)
        break
    except Exception as e:
        print("gagal versi", v, "->", str(e)[:150])

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to ../dataset_denny_v2 in yolov8:: 100%|██████████| 1186/1186 [00:01<00:00, 864.06it/s]

OK versi 2 c:\Users\riizk\Documents\3.Collage\SKRIPSI\dataset_denny_v2


: 

## 9. Dataset pembanding 2 (Aldian)
Download → `../dataset_aldian_v1`. Dipakai sebagai sumber data tambahan pada penggabungan dataset (bagian 11).

In [1]:
import os, random, cv2
from pathlib import Path
from dotenv import load_dotenv
from roboflow import Roboflow

load_dotenv()
rf = Roboflow(api_key=os.getenv("ROBOFLOW_API_KEY"))
proj = rf.workspace("aldians-workspace").project("indonesian-plate-number-mpgfj-npmds")
ds3 = proj.version(1).download("yolov8", location="../dataset_aldian_v1")
print(ds3.location)

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to ../dataset_aldian_v1 in yolov8:: 100%|██████████| 5543/5543 [00:05<00:00, 952.18it/s] 


c:\Users\riizk\Documents\3.Collage\SKRIPSI\dataset_aldian_v1


## 10. Cek label & statistik dataset Aldian
Visualisasi sampel label (hasil gambar `cek_label_aldian` sudah dihapus), histogram lebar plat, dan jumlah gambar yang lolos filter lebar plat 0.03–0.35.

> Sel pertama di bagian ini menulis gambar ke ../cek_label_aldian yang sudah dihapus. **Jangan dijalankan ulang.**

In [2]:
root3 = Path(ds3.location).resolve()
for s in ("train", "valid", "test"):
    d = root3 / s / "images"
    print(s, len(list(d.glob("*"))) if d.exists() else "tidak ada")
print((root3 / "data.yaml").read_text())

split3 = next(s for s in ("train", "valid", "test") if (root3 / s / "images").exists())
files3 = sorted((root3 / split3 / "images").iterdir())
random.seed(1)
cek3 = Path("../cek_label_aldian")
cek3.mkdir(exist_ok=True)

for p in random.sample(files3, min(12, len(files3))):
    img = cv2.imread(str(p))
    H, W = img.shape[:2]
    lbl = root3 / split3 / "labels" / (p.stem + ".txt")
    for line in (lbl.read_text().splitlines() if lbl.exists() else []):
        v = [float(x) for x in line.split()[1:]]
        if len(v) == 4:
            xc, yc, bw, bh = v
            x1, x2 = (xc - bw / 2) * W, (xc + bw / 2) * W
            y1, y2 = (yc - bh / 2) * H, (yc + bh / 2) * H
        else:
            xs, ys = v[0::2], v[1::2]
            x1, x2, y1, y2 = min(xs) * W, max(xs) * W, min(ys) * H, max(ys) * H
        cv2.rectangle(img, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), max(3, W // 300))
    sc = 1000 / max(H, W)
    cv2.imwrite(str(cek3 / (p.stem[:25] + ".jpg")), cv2.resize(img, None, fx=sc, fy=sc, interpolation=cv2.INTER_AREA))
print("selesai ->", cek3.resolve())

train 1927
valid 566
test 276
names:
- plat-nomor
nc: 1
roboflow:
  license: CC BY 4.0
  project: indonesian-plate-number-mpgfj-npmds
  url: https://universe.roboflow.com/aldians-workspace/indonesian-plate-number-mpgfj-npmds/dataset/1
  version: 1
  workspace: aldians-workspace
test: ../test/images
train: ../train/images
val: ../valid/images

selesai -> C:\Users\riizk\Documents\3.Collage\SKRIPSI\cek_label_aldian


In [3]:
import numpy as np

rows = []
for s in ("train", "valid", "test"):
    d = root3 / s / "labels"
    if not d.exists():
        continue
    for lf in d.glob("*.txt"):
        for line in lf.read_text().splitlines():
            v = [float(x) for x in line.split()[1:]]
            if len(v) == 4:
                w, h = v[2], v[3]
            else:
                xs, ys = v[0::2], v[1::2]
                w, h = max(xs) - min(xs), max(ys) - min(ys)
            rows.append((s, lf.stem, w, h))

w = np.array([r[2] for r in rows])
print("total kotak:", len(w))
for s in ("train", "valid", "test"):
    print(s, "kotak:", sum(1 for r in rows if r[0] == s))

bins = [0, 0.05, 0.1, 0.2, 0.35, 0.6, 1.01]
cnt, _ = np.histogram(w, bins=bins)
for lo, hi, c in zip(bins[:-1], bins[1:], cnt):
    print(f"lebar plat {lo:.2f}-{min(hi, 1):.2f} dari lebar gambar: {c} ({100 * c / len(w):.0f}%)")

total kotak: 3458
train kotak: 2425
valid kotak: 714
test kotak: 319
lebar plat 0.00-0.05 dari lebar gambar: 162 (5%)
lebar plat 0.05-0.10 dari lebar gambar: 475 (14%)
lebar plat 0.10-0.20 dari lebar gambar: 610 (18%)
lebar plat 0.20-0.35 dari lebar gambar: 441 (13%)
lebar plat 0.35-0.60 dari lebar gambar: 394 (11%)
lebar plat 0.60-1.00 dari lebar gambar: 1373 (40%)


In [4]:
from collections import Counter

print((root3 / "data.yaml").read_text())

per_img = {}
cls_count = Counter()
for s in ("train", "valid", "test"):
    d = root3 / s / "labels"
    if not d.exists():
        continue
    for lf in d.glob("*.txt"):
        boxes = []
        for line in lf.read_text().splitlines():
            parts = line.split()
            cls_count[parts[0]] += 1
            v = [float(x) for x in parts[1:]]
            boxes.append(v[2] if len(v) == 4 else max(v[0::2]) - min(v[0::2]))
        per_img[(s, lf.stem)] = boxes

print("kelas (id: jumlah kotak):", dict(cls_count))
LO, HI = 0.03, 0.35
for s in ("train", "valid", "test"):
    items = [(k, b) for k, b in per_img.items() if k[0] == s]
    keep = [k for k, b in items if b and all(LO <= x <= HI for x in b)]
    print(s, "gambar total:", len(items), "| lolos filter:", len(keep))
    

names:
- plat-nomor
nc: 1
roboflow:
  license: CC BY 4.0
  project: indonesian-plate-number-mpgfj-npmds
  url: https://universe.roboflow.com/aldians-workspace/indonesian-plate-number-mpgfj-npmds/dataset/1
  version: 1
  workspace: aldians-workspace
test: ../test/images
train: ../train/images
val: ../valid/images

kelas (id: jumlah kotak): {'0': 3458}
train gambar total: 1927 | lolos filter: 646
valid gambar total: 566 | lolos filter: 188
test gambar total: 276 | lolos filter: 96


## 11. Penggabungan dataset
Gabung `dataset_roboflow` + data Aldian terfilter → `../dataset_gabungan`; sisa test Aldian → `../eval_aldian_test`.

In [5]:
import shutil, cv2
from pathlib import Path

base = Path("..").resolve()
ali = base / "dataset_roboflow"
ald = base / "dataset_aldian_v1"
out = base / "dataset_gabungan"
ev = base / "eval_aldian_test"
LO, HI, MAXSIDE = 0.03, 0.35, 1280

def to_box(parts):
    v = [float(x) for x in parts[1:]]
    if len(v) == 4:
        return v
    xs, ys = v[0::2], v[1::2]
    return [(min(xs) + max(xs)) / 2, (min(ys) + max(ys)) / 2,
            max(xs) - min(xs), max(ys) - min(ys)]

def prep_dirs(root, splits):
    for s in splits:
        (root / s / "images").mkdir(parents=True, exist_ok=True)
        (root / s / "labels").mkdir(parents=True, exist_ok=True)

def write_label(path, boxes):
    path.write_text("".join(f"0 {b[0]:.6f} {b[1]:.6f} {b[2]:.6f} {b[3]:.6f}\n" for b in boxes))

prep_dirs(out, ("train", "valid"))
prep_dirs(ev, ("test",))

n_ali = {}
for s in ("train", "valid"):
    n = 0
    for img in (ali / s / "images").iterdir():
        lbl = ali / s / "labels" / (img.stem + ".txt")
        boxes = [to_box(l.split()) for l in lbl.read_text().splitlines()] if lbl.exists() else []
        shutil.copy2(img, out / s / "images" / ("jalan_" + img.name))
        write_label(out / s / "labels" / ("jalan_" + img.stem + ".txt"), boxes)
        n += 1
    n_ali[s] = n

n_ald = {}
for s in ("train", "valid", "test"):
    dest = ev / "test" if s == "test" else out / s
    by_stem = {p.stem: p for p in (ald / s / "images").iterdir()}
    n = 0
    for lbl in (ald / s / "labels").glob("*.txt"):
        boxes = [to_box(l.split()) for l in lbl.read_text().splitlines()]
        if not boxes or not all(LO <= b[2] <= HI for b in boxes):
            continue
        src = by_stem.get(lbl.stem)
        if src is None:
            continue
        im = cv2.imread(str(src))
        if im is None:
            continue
        sc = MAXSIDE / max(im.shape[:2])
        if sc < 1:
            im = cv2.resize(im, None, fx=sc, fy=sc, interpolation=cv2.INTER_AREA)
        name = "aldian_" + lbl.stem
        cv2.imwrite(str(dest / "images" / (name + ".jpg")), im, [cv2.IMWRITE_JPEG_QUALITY, 95])
        write_label(dest / "labels" / (name + ".txt"), boxes)
        n += 1
    n_ald[s] = n

(out / "data.yaml").write_text(
    f"path: {out.as_posix()}\ntrain: train/images\nval: valid/images\nnc: 1\nnames: ['plate']\n")
(ev / "data.yaml").write_text(
    f"path: {ev.as_posix()}\ntrain: test/images\nval: test/images\ntest: test/images\nnc: 1\nnames: ['plate']\n")

print("kamera jalan:", n_ali, "| aldian terpilih:", n_ald)
for s in ("train", "valid"):
    print("gabungan", s, len(list((out / s / "images").iterdir())))
print("eval aldian test:", len(list((ev / "test" / "images").iterdir())))

kamera jalan: {'train': 1098, 'valid': 167} | aldian terpilih: {'train': 646, 'valid': 188, 'test': 96}
gabungan train 1744
gabungan valid 355
eval aldian test: 96


## 12. Evaluasi model lama (baseline) pada dua test set
Test kamera jalan vs test Aldian terfilter. Hasil baseline di Aldian rendah (mAP50 0.190) → alasan menggabungkan dataset.

In [6]:
from ultralytics import YOLO
from pathlib import Path

best0 = Path("../runs/slpr_v8n_run1/weights/best.pt").resolve()
runs_dir = str(best0.parents[2])
base_model = YOLO(str(best0))

tes = {
    "kamera_jalan_test": Path("../dataset_roboflow/data_fixed.yaml").resolve(),
    "aldian_test_terfilter": Path("../eval_aldian_test/data.yaml").resolve(),
}
for nama, yaml_path in tes.items():
    m = base_model.val(data=str(yaml_path), split="test", imgsz=640, device=0,
                       project=runs_dir, name="baseline_" + nama, verbose=False)
    print(f"{nama}: P={m.box.mp:.3f} R={m.box.mr:.3f} mAP50={m.box.map50:.3f} mAP50-95={m.box.map:.3f}")

Ultralytics 8.4.173  Python-3.14.5 torch-2.14.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 51.814.6 MB/s, size: 32.4 KB)
val: Scanning C:\Users\riizk\Documents\3.Collage\SKRIPSI\dataset_roboflow\test\labels.cache... 54 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 54/54 11.3Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 1.0s/it 4.0s0.4s9s
                   all         54         54      0.998      0.981      0.995      0.688
Speed: 3.0ms preprocess, 7.2ms inference, 0.0ms loss, 4.3ms postprocess per image
Results saved to C:\Users\riizk\Documents\3.Collage\SKRIPSI\runs\baseline_kamera_jalan_test
kamera_jalan_test: P=0.998 R=0.981 mAP50=0.995 mAP50-95=0.688
Ultralytics 8.4.173  Python-3.14.5 torch-2.14.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
M

## 13. Training ulang pada dataset gabungan
Fine-tune dari baseline, 80 epoch → `../runs/slpr_v8n_gabungan_run1`.

In [7]:
model = YOLO(str(best0))
model.train(
    data=str(Path("../dataset_gabungan/data.yaml").resolve()),
    epochs=80,
    imgsz=640,
    batch=16,
    patience=20,
    fliplr=0.0,
    device=0,
    workers=2,
    project=runs_dir,
    name="slpr_v8n_gabungan_run1",
)


Ultralytics 8.4.173  Python-3.14.5 torch-2.14.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\riizk\Documents\3.Collage\SKRIPSI\dataset_gabungan\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=80, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=C:\Users\riizk\Documents\3.Collage\SKRIPSI\runs\slpr_v8n_r

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x0000021EADC08600>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0480

## 14. Evaluasi model baru & uji foto asli
Bandingkan model lama vs baru pada kedua test set, lalu prediksi `../foto_uji` (foto pribadi).

In [8]:
best1 = Path(runs_dir) / "slpr_v8n_gabungan_run1" / "weights" / "best.pt"
print("best.pt baru ada:", best1.exists())
new_model = YOLO(str(best1))

for nama, yaml_path in tes.items():
    m = new_model.val(data=str(yaml_path), split="test", imgsz=640, device=0,
                      project=runs_dir, name="gabungan_" + nama, verbose=False)
    print(f"{nama}: P={m.box.mp:.3f} R={m.box.mr:.3f} mAP50={m.box.map50:.3f} mAP50-95={m.box.map:.3f}")

foto_dir = Path("../foto_uji").resolve()
for p in sorted(foto_dir.iterdir()):
    if p.suffix.lower() not in (".jpg", ".jpeg", ".png"):
        continue
    for nama, mdl in (("lama", base_model), ("baru", new_model)):
        r = mdl.predict(str(p), conf=0.25, device=0, verbose=False)[0]
        print(p.name[-18:], nama, "-> deteksi:", len(r.boxes),
              [round(float(c), 2) for c in r.boxes.conf.cpu().numpy()])
              

best.pt baru ada: True
Ultralytics 8.4.173  Python-3.14.5 torch-2.14.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 80.515.3 MB/s, size: 30.9 KB)
val: Scanning C:\Users\riizk\Documents\3.Collage\SKRIPSI\dataset_roboflow\test\labels.cache... 54 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 54/54 25.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 1.3it/s 3.2s0.3ss
                   all         54         54      0.982      0.999      0.995      0.695
Speed: 2.4ms preprocess, 8.1ms inference, 0.0ms loss, 1.3ms postprocess per image
Results saved to C:\Users\riizk\Documents\3.Collage\SKRIPSI\runs\gabungan_kamera_jalan_test
kamera_jalan_test: P=0.982 R=0.999 mAP50=0.995 mAP50-95=0.695
Ultralytics 8.4.173  Python-3.14.5 torch-2.14.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4050 

In [9]:
from pathlib import Path
from ultralytics import YOLO

best1 = Path("../runs/slpr_v8n_gabungan_run1/weights/best.pt").resolve()
print("best.pt ada:", best1.exists())

onnx_new = YOLO(str(best1)).export(format="onnx", imgsz=640, simplify=True)
print(onnx_new)

tes = {
    "kamera_jalan_test": Path("../dataset_roboflow/data_fixed.yaml").resolve(),
    "aldian_test_terfilter": Path("../eval_aldian_test/data.yaml").resolve(),
}
m_onnx = YOLO(str(onnx_new), task="detect")
for nama, y in tes.items():
    m = m_onnx.val(data=str(y), split="test", imgsz=640, device="cpu", batch=1,
                   project=str(best1.parents[2]), name="onnx_gabungan_" + nama, verbose=False)
    print(f"{nama}: P={m.box.mp:.3f} R={m.box.mr:.3f} mAP50={m.box.map50:.3f} mAP50-95={m.box.map:.3f}")

best.pt ada: True
Ultralytics 8.4.173  Python-3.14.5 torch-2.14.1+cu126 CPU (AMD Ryzen 7 8845HS w/ Radeon 780M Graphics)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from 'C:\Users\riizk\Documents\3.Collage\SKRIPSI\runs\slpr_v8n_gabungan_run1\weights\best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 5, 8400) (5.9 MB)

ONNX: starting export with onnx 1.23.1 opset 18...
ONNX: slimming with onnxslim 0.1.97...
ONNX: export success  2.1s, saved as 'C:\Users\riizk\Documents\3.Collage\SKRIPSI\runs\slpr_v8n_gabungan_run1\weights\best.onnx' (11.7 MB)

Export complete (2.5s)
Results saved to C:\Users\riizk\Documents\3.Collage\SKRIPSI\runs\slpr_v8n_gabungan_run1\weights\best.onnx
Predict:         yolo predict task=detect model=C:\Users\riizk\Documents\3.Collage\SKRIPSI\runs\slpr_v8n_gabungan_run1\weights\best.onnx imgsz=640 
Validate:        yolo val task=detect model=C:\Users\riizk\Documents\3.Collage\SKRIPSI\runs\slpr_v

In [10]:
m = YOLO(str(best1)).val(data=str(tes["aldian_test_terfilter"]), split="test", imgsz=640,
                         device="cpu", batch=1, project=str(best1.parents[2]),
                         name="pt_cpu_gabungan_aldian", verbose=False)
print(f"PT CPU Aldian: P={m.box.mp:.3f} R={m.box.mr:.3f} mAP50={m.box.map50:.3f} mAP50-95={m.box.map:.3f}")

Ultralytics 8.4.173  Python-3.14.5 torch-2.14.1+cu126 CPU (AMD Ryzen 7 8845HS w/ Radeon 780M Graphics)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 1913.2707.9 MB/s, size: 280.1 KB)
val: Scanning C:\Users\riizk\Documents\3.Collage\SKRIPSI\eval_aldian_test\test\labels.cache... 96 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 96/96 33.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 96/96 29.6it/s 3.2s0.0s
                   all         96        150      0.918      0.947      0.959      0.759
Speed: 0.3ms preprocess, 26.9ms inference, 0.0ms loss, 0.6ms postprocess per image
Results saved to C:\Users\riizk\Documents\3.Collage\SKRIPSI\runs\pt_cpu_gabungan_aldian
PT CPU Aldian: P=0.918 R=0.947 mAP50=0.959 mAP50-95=0.759
